# 3/7 - D. WHEN - reaction speed and possession-level pressure
Time-to-press after losing the ball, counter-press fast / slow / never, zone-weighted intensity, possession events, co-press, intensity over time, pressure -> regain.

*Original sections: sections 8, 8b, 9, 12, 14.*  Barca (team 0) vs Atletico (team 1), first half.

**Run the notebooks in order** (each one reads what the earlier ones saved in `new_cache/pressure_analysis/state/`):

| # | notebook | what it covers |
|---|---|---|
| 1 | 01_pressure_signal.ipynb | B. The pressure signal |
| 2 | 02_where_zone_channel_line.ipynb | C. WHERE - zone, channel, defensive line, shape |
| 3 | **03_timing_and_possession.ipynb** | D. WHEN - reaction speed and possession-level pressure |
| 4 | 04_passes_outcomes_triggers.ipynb | E/F. EFFECT and TRIGGERS |
| 5 | 05_summary_and_registry.ipynb | G/H. Summary + event registry |
| 6 | 06_clips.ipynb | H. Clips |
| 7 | 07_exports.ipynb | I. Exports |

Shared setup (paths, tables, data contract, team-label orientation, helpers) lives in `pressure_common.py` - keep it in the same folder as the notebooks.

In [ ]:
from pressure_common import *

In [ ]:
# handed over by the earlier notebooks:
#   01_pressure_signal: STOPPAGE_GAP_SEC, out_of_possession_seconds, stoppage_frame_lookup, team_pressure_events, team_pressure_full, tpf
#   02_where_zone_channel_line: tpf_zone
load_state(globals(), [
    'STOPPAGE_GAP_SEC',
    'out_of_possession_seconds',
    'stoppage_frame_lookup',
    'team_pressure_events',
    'team_pressure_full',
    'tpf',
    'tpf_zone',
])

## 8. Time-to-press after losing the ball
Turnovers derived from raw (unsmoothed) carrier-team changes, filtered to stable holds (>=1.5s) to exclude 50/50 scramble noise.

In [ ]:
MIN_HOLD_SEC = 1.5
MIN_HOLD_FRAMES = int(MIN_HOLD_SEC * FPS)
MAX_SEARCH_WINDOW_SEC = 10
MAX_SEARCH_WINDOW_FRAMES = MAX_SEARCH_WINDOW_SEC * FPS

raw_carrier = ball_frame_table.dropna(subset=["carrier_team"])[["frame_idx", "carrier_team"]].sort_values("frame_idx").reset_index(drop=True)
raw_carrier["state_change"] = (raw_carrier["carrier_team"] != raw_carrier["carrier_team"].shift()).cumsum()
carrier_runs = (
    raw_carrier.groupby("state_change")
    .agg(start_frame=("frame_idx", "first"), end_frame=("frame_idx", "last"), n_frames=("frame_idx", "count"), carrier_team=("carrier_team", "first"))
    .reset_index(drop=True)
)

stable_runs = carrier_runs[carrier_runs["n_frames"] >= MIN_HOLD_FRAMES].reset_index(drop=True)

turnovers = []
for i in range(len(stable_runs) - 1):
    losing_team = stable_runs.loc[i, "carrier_team"]
    gaining_team = stable_runs.loc[i + 1, "carrier_team"]
    if losing_team != gaining_team:
        turnovers.append({
            "losing_team": losing_team,
            "loss_frame": stable_runs.loc[i, "end_frame"],
            "gain_frame": stable_runs.loc[i + 1, "start_frame"],   # opponent's confirmed takeover = the real turnover boundary
        })
turnovers = pd.DataFrame(turnovers)

press_lookup = team_pressure_full[["frame_idx", "team", "under_team_pressure"]].copy()

results = []
for _, row in turnovers.iterrows():
    team_id = row["losing_team"]
    loss_frame, gain_frame = row["loss_frame"], row["gain_frame"]
    # anchored on gain_frame (same anchor as the regain window in 8b), not on the losing team's last touch
    window = press_lookup[
        (press_lookup["team"] == team_id) &
        (press_lookup["frame_idx"] >= gain_frame) &
        (press_lookup["frame_idx"] <= gain_frame + MAX_SEARCH_WINDOW_FRAMES) &
        (press_lookup["under_team_pressure"])
    ]
    if len(window) > 0:
        first_press_frame = window["frame_idx"].min()
        results.append({"team": team_id, "loss_frame": loss_frame, "gain_frame": gain_frame,
                        "time_to_press_sec": (first_press_frame - gain_frame) / FPS})

time_to_press = pd.DataFrame(results)

# --- stable-hold regain helper, shared by 8b / 8c / 14 --------------------------------------
# A regain only counts if the team starts a stable hold (>= MIN_HOLD_SEC), the same noise filter that
# defines turnovers. A single stray raw carrier frame no longer counts as a regain.
_sr_start = stable_runs["start_frame"].values
_sr_team = stable_runs["carrier_team"].values

def stable_regain(team_id, after_frame, until_frame):
    """True if team_id starts a stable hold with start_frame in (after_frame, until_frame]."""
    lo = np.searchsorted(_sr_start, after_frame, side="right")
    hi = np.searchsorted(_sr_start, until_frame, side="right")
    return bool(np.any(_sr_team[lo:hi] == team_id))

print("genuine turnover events:", len(turnovers))
print(f"turnovers with a press detected within {MAX_SEARCH_WINDOW_SEC}s: {len(time_to_press)} / {len(turnovers)}")
print("\ntime-to-press stats by team (seconds):")
print(time_to_press.groupby("team")["time_to_press_sec"].agg(["mean", "median", "min", "max", "count"]))


## 8b. Counter-press effectiveness -- time-to-press by response speed
Extends section 8: does responding fast to losing the ball actually predict winning it back, or is that just what pressure volume looks like anyway? Turnovers are re-anchored on the opponent's *confirmed* takeover frame (the fix below), not the losing team's own last touch -- anchoring on the losing team's own frame trivially matched `losing_team` in the regain window every time, which is what the corrected version fixes. `time_to_press_sec` (from section 8) is then bucketed into fast (pressed within `COUNTER_PRESS_WINDOW_SEC`) / slow (pressed later, within section 8's `MAX_SEARCH_WINDOW_SEC`) / never.

In [ ]:
## Counter-press effectiveness — corrected anchor point
## Bug in prior version: loss_frame is the LAST frame of the losing team's own run, so the window's
## own starting frame trivially matched losing_team every time (100% regain rate for both teams was
## the tell). Fixed by anchoring on the confirmed start of the opponent's next stable run instead.

COUNTER_PRESS_WINDOW_SEC = 5.0
COUNTER_PRESS_WINDOW_FRAMES = int(COUNTER_PRESS_WINDOW_SEC * FPS)

# local carrier lookup -- kept separate from section 14's _carrier_frame_idx/_carrier_team_vals
# (same underlying data, different variable names) so this section runs standalone regardless
# of where it sits relative to section 14
_cp_carrier_lookup = (
    ball_frame_table.dropna(subset=["carrier_team"])[["frame_idx", "carrier_team"]]
    .sort_values("frame_idx")
)
_cp_carrier_frame_idx = _cp_carrier_lookup["frame_idx"].values
_cp_carrier_team_vals = _cp_carrier_lookup["carrier_team"].values

# rebuild turnovers with the opponent's confirmed start frame included
turnovers_fixed = []
for i in range(len(stable_runs) - 1):
    losing_team = stable_runs.loc[i, "carrier_team"]
    gaining_team = stable_runs.loc[i + 1, "carrier_team"]
    if losing_team != gaining_team:
        turnovers_fixed.append({
            "losing_team": losing_team,
            "loss_frame": stable_runs.loc[i, "end_frame"],
            "gain_frame": stable_runs.loc[i + 1, "start_frame"],  # opponent's confirmed takeover, the real turnover boundary
        })
turnovers_fixed = pd.DataFrame(turnovers_fixed)

def regained_after_turnover(losing_team, gain_frame):
    # strictly AFTER the opponent's takeover, and only a stable hold counts (see stable_regain in section 8)
    return stable_regain(losing_team, gain_frame, gain_frame + COUNTER_PRESS_WINDOW_FRAMES)

turnovers_fixed["regained_within_window"] = turnovers_fixed.apply(
    lambda r: regained_after_turnover(r["losing_team"], r["gain_frame"]), axis=1
)

turnovers_annotated = turnovers_fixed.merge(
    time_to_press.rename(columns={"team": "losing_team"}),
    on=["losing_team", "loss_frame", "gain_frame"],
    how="left"
)
turnovers_annotated["pressed_fast"] = turnovers_annotated["time_to_press_sec"] <= COUNTER_PRESS_WINDOW_SEC

print("=" * 60)
print("COUNTER-PRESS EFFECTIVENESS")
print("=" * 60)
print(f"{len(turnovers_fixed)} turnovers, checking regain within {COUNTER_PRESS_WINDOW_SEC}s of the opponent's confirmed takeover")

print("\ncounter-press regain rate by team (regained the ball back within window, regardless of press):")
regain_by_team = turnovers_annotated.groupby("losing_team")["regained_within_window"].agg(["mean", "sum", "count"])
regain_by_team["mean"] = (regain_by_team["mean"] * 100).round(1)
print(regain_by_team.rename(columns={"mean": "regain_rate_pct"}))

print(f"\ndoes pressing fast (within {COUNTER_PRESS_WINDOW_SEC}s) actually correlate with winning it back?")
print("regain rate split by whether a press was mounted within the window:")
cross = turnovers_annotated.groupby(["losing_team", "pressed_fast"])["regained_within_window"].agg(["mean", "count"])
cross["mean"] = (cross["mean"] * 100).round(1)
print(cross.rename(columns={"mean": "regain_rate_pct"}))

In [ ]:
## Counter-press effectiveness — 3-way press-response split (fast / slow / never)
## DESCRIPTIVE ONLY: this split is biased towards "fast" (a quick regain stops the opponent-carrier frames
## that a press needs, so quick regains get filed under slow/never). Use the landmark table below for any
## claim that fast pressing predicts regaining the ball.
## Replaces the earlier pressed_fast boolean, which folded "pressed slowly" and "never pressed"
## into the same False bucket (NaN <= 5.0 evaluates to False in pandas). These are different claims
## and get separate rows here.

def classify_press_response(t):
    if pd.isna(t):
        return "never"
    elif t <= COUNTER_PRESS_WINDOW_SEC:
        return "fast"
    else:
        return "slow"

turnovers_annotated["press_response"] = turnovers_annotated["time_to_press_sec"].apply(classify_press_response)
turnovers_annotated["press_response"] = pd.Categorical(
    turnovers_annotated["press_response"], categories=["fast", "slow", "never"], ordered=True
)

# TEAM_NAMES comes from Setup-2 (derived from the table's team_name column)

counter_press_summary = (
    turnovers_annotated.groupby(["losing_team", "press_response"], observed=False)["regained_within_window"]
    .agg(regains="sum", turnovers="count")
    .reset_index()
)
counter_press_summary["regain_rate_pct"] = (counter_press_summary["regains"] / counter_press_summary["turnovers"] * 100).round(1)

print("=" * 70)
print("COUNTER-PRESS EFFECTIVENESS BY PRESS-RESPONSE SPEED")
print("=" * 70)
print(f"fast  = pressed within {COUNTER_PRESS_WINDOW_SEC:.0f}s of the opponent's confirmed takeover")
print(f"slow  = pressed between {COUNTER_PRESS_WINDOW_SEC:.0f}-{MAX_SEARCH_WINDOW_SEC}s (Section 8's search window)")
print(f"never = no press detected within Section 8's {MAX_SEARCH_WINDOW_SEC}s window at all")
print()

for team_id, team_name in TEAM_NAMES.items():
    print(f"{team_name} (team {team_id})")
    print("-" * 40)
    team_rows = (
        counter_press_summary[counter_press_summary["losing_team"] == team_id]
        .set_index("press_response")
        .reindex(["fast", "slow", "never"])
    )
    for response, row in team_rows.iterrows():
        n = 0 if pd.isna(row["turnovers"]) else int(row["turnovers"])
        if n == 0:
            print(f"  {response:<6} : n/a (0 turnovers)")
            continue
        regains = int(row["regains"])
        rate = row["regain_rate_pct"]
        print(f"  {response:<6} : {rate:5.1f}% regain rate  ({regains}/{n} turnovers)")
    print()

print("=" * 70)
print("OVERALL (both teams combined)")
print("=" * 70)
overall = (
    turnovers_annotated.groupby("press_response", observed=False)["regained_within_window"]
    .agg(regains="sum", turnovers="count")
    .reindex(["fast", "slow", "never"])
)
overall["regain_rate_pct"] = (overall["regains"] / overall["turnovers"] * 100).round(1)
for response, row in overall.iterrows():
    n = int(row["turnovers"])
    print(f"  {response:<6} : {row['regain_rate_pct']:5.1f}% regain rate  ({int(row['regains'])}/{n} turnovers)")

In [ ]:
## Counter-press effectiveness -- LANDMARK version (removes the immortal-time bias of the fast/slow/never split)
## Take a fixed landmark LANDMARK_SEC after the opponent's takeover and keep only turnovers where the losing team
## has NOT yet regained the ball. Everyone in the sample is still "at risk" of regaining, so compare:
##   pressed by the landmark  vs  not pressed by the landmark
## and measure the regain only AFTER the landmark, over REGAIN_HORIZON_SEC.
LANDMARK_SEC = COUNTER_PRESS_WINDOW_SEC
LANDMARK_FRAMES = int(LANDMARK_SEC * FPS)
REGAIN_HORIZON_SEC = 5.0
REGAIN_HORIZON_FRAMES = int(REGAIN_HORIZON_SEC * FPS)

lm = turnovers_annotated.copy()
lm["regained_before_landmark"] = [
    stable_regain(t, g, g + LANDMARK_FRAMES) for t, g in zip(lm["losing_team"], lm["gain_frame"])
]
n_all = len(lm)
lm = lm[~lm["regained_before_landmark"]].copy()
lm["pressed_by_landmark"] = lm["time_to_press_sec"] <= LANDMARK_SEC          # NaN -> False
lm["regained_after_landmark"] = [
    stable_regain(t, g + LANDMARK_FRAMES, g + LANDMARK_FRAMES + REGAIN_HORIZON_FRAMES)
    for t, g in zip(lm["losing_team"], lm["gain_frame"])
]

counter_press_landmark = (
    lm.groupby(["losing_team", "pressed_by_landmark"])["regained_after_landmark"]
    .agg(regains="sum", turnovers="count").reset_index()
)
counter_press_landmark["regain_rate_pct"] = (counter_press_landmark["regains"] / counter_press_landmark["turnovers"] * 100).round(1)
_ov = lm.groupby("pressed_by_landmark")["regained_after_landmark"].agg(regains="sum", turnovers="count")
_ov["regain_rate_pct"] = (_ov["regains"] / _ov["turnovers"] * 100).round(1)

print(f"{len(lm)} / {n_all} turnovers still un-regained at the {LANDMARK_SEC:.0f}s landmark")
print(f"regain within the next {REGAIN_HORIZON_SEC:.0f}s, by whether a press was mounted by the landmark:\n")
print(counter_press_landmark.to_string(index=False))
print("\nboth teams combined:\n", _ov)
print("\n(small n per cell -- report counts alongside the rates)")


## 9. Zone-weighted pressing intensity
Attacking-third pressure counts more than own-third pressure (own=1x, middle=2x, attacking=3x), since winning the ball high up the pitch is tactically more valuable. Score is bounded 0-100% by normalizing against the theoretical max (every out-of-possession frame spent pressing in the attacking third).

In [ ]:
ZONE_WEIGHTS = {"own_third": 1, "middle_third": 2, "attacking_third": 3}

weighted = tpf_zone[tpf_zone["under_team_pressure"] & tpf_zone["press_zone"].notna()].copy()
weighted["zone_weight"] = weighted["press_zone"].map(ZONE_WEIGHTS)

weighted_pressure_frames = weighted.groupby("team")["zone_weight"].sum()
weighted_pressure_seconds = weighted_pressure_frames / FPS

print("weighted pressure seconds by team (own=1x, middle=2x, attacking=3x):\n", weighted_pressure_seconds)

print("\nzone-weighted pressing intensity, bounded 0-100%:")
for team_id in [0, 1]:
    theoretical_max_seconds = out_of_possession_seconds[team_id] * ZONE_WEIGHTS["attacking_third"]
    pct_of_max = weighted_pressure_seconds.get(team_id, 0) / theoretical_max_seconds * 100
    print(f"team {team_id}: {pct_of_max:.1f}% of theoretical max weighted pressing")


## 12. Possession events (keep-last gap fill) + pressure per event
Match is split into possession events by forward-filling every no-carrier (NONE) gap onto the last known `possession_team`, then run-length-encoding the result. For each event, the out-of-possession team's frame-level pressure signal (section 3) is averaged over that event's frames; the final per-team number is a duration-weighted mean across events, so a 40s press counts more than a 2s one.

In [ ]:
poss_raw = ball_frame_table[["frame_idx", "possession_team"]].sort_values("frame_idx").reset_index(drop=True)
poss_raw = poss_raw.merge(stoppage_frame_lookup.rename("is_stoppage"), left_on="frame_idx", right_index=True, how="left")
poss_raw["is_stoppage"] = poss_raw["is_stoppage"].fillna(False)

poss_raw["possession_team_filled"] = poss_raw["possession_team"].ffill()
poss_raw = poss_raw.dropna(subset=["possession_team_filled"]).reset_index(drop=True)
poss_raw["possession_team_filled"] = poss_raw["possession_team_filled"].astype(int)

# force a hard break at every stoppage -- even if the same team has the ball before and after,
# dead time should never let two possession events merge into one
poss_raw["grouping_state"] = np.where(poss_raw["is_stoppage"], -1, poss_raw["possession_team_filled"])
poss_raw["event_id"] = (poss_raw["grouping_state"] != poss_raw["grouping_state"].shift()).cumsum()

# drop the stoppage rows themselves -- dead time shouldn't count toward any event's duration,
# but the split they forced above is preserved
poss_raw_live = poss_raw[~poss_raw["is_stoppage"]].reset_index(drop=True)

possession_events = (
    poss_raw_live.groupby("event_id", as_index=False)
    .agg(
        possession_team=("possession_team_filled", "first"),
        start_frame=("frame_idx", "first"),
        end_frame=("frame_idx", "last"),
        n_frames=("frame_idx", "count"),
    )
)
possession_events["duration_sec"] = possession_events["n_frames"] / FPS
possession_events["pressing_team"] = 1 - possession_events["possession_team"]

print(f"{len(possession_events)} possession events (keep-last gap fill, split at stoppages >= {STOPPAGE_GAP_SEC}s)")
print(possession_events["duration_sec"].describe())

In [ ]:
frame_to_event = poss_raw_live[["frame_idx", "event_id", "possession_team_filled"]]

tpf_events = tpf.merge(frame_to_event, on="frame_idx", how="inner")
# keep only rows where tpf's pressing team actually matches this event's out-of-possession team --
# drops the rare frames where raw carrier_team briefly disagreed with the smoothed possession_team
tpf_events = tpf_events[tpf_events["team"] != tpf_events["possession_team_filled"]].copy()

event_pressure = (
    tpf_events.groupby("event_id", as_index=False)
    .agg(
        mean_n_pressing=("n_pressing", "mean"),
        mean_frac_tight_blocked=("frac_tight_blocked", "mean"),
        pct_frames_under_pressure=("under_team_pressure", "mean"),
        n_frames_matched=("frame_idx", "count"),
    )
)

possession_events = possession_events.merge(event_pressure, on="event_id", how="left")
matched = possession_events["pct_frames_under_pressure"].notna().sum()
print(f"pressure computed for {matched} / {len(possession_events)} events "
      f"(unmatched events are ones where raw carrier_team never agreed with the filled possession_team)")

scored = possession_events.dropna(subset=["pct_frames_under_pressure"])

final_pressure_by_team = (
    scored.groupby("pressing_team")
    .apply(lambda g: pd.Series({
        "duration_weighted_pct_under_pressure": np.average(g["pct_frames_under_pressure"], weights=g["duration_sec"]) * 100,
        "simple_mean_pct_under_pressure": g["pct_frames_under_pressure"].mean() * 100,
        "n_events": len(g),
        "total_duration_sec": g["duration_sec"].sum(),
    }))
)
print("\nfinal pressure by pressing team (duration-weighted mean is the headline number):")
print(final_pressure_by_team)

In [ ]:
# --- the three "pressing %" definitions side by side (pick ONE as the thesis headline) ---
_ev_frames = team_pressure_events.groupby("team")["n_frames"].sum().rename(index=int)
pressing_pct_reconciliation = pd.DataFrame({
    "A_event_secs_over_OOP_secs": pd.Series({t: _ev_frames.get(t, 0) / FPS / out_of_possession_seconds[t] * 100 for t in [0, 1]}),
    "B_duration_weighted_possession_events": final_pressure_by_team["duration_weighted_pct_under_pressure"].rename(index=int),
    "C_share_of_carrier_frames": (tpf.groupby("team")["under_team_pressure"].mean() * 100).rename(index=int),
}).round(1)
print("pressing % under the three denominators (A = notebook-1 section 4, B = section 12, C = section 14 bins):")
print(pressing_pct_reconciliation)


## 14. Pressure effectiveness — co-press count, intensity over time, regain rate
Three cheap additions on top of the already-validated pressure signal (section 3) and raw carrier data (section 8): how many defenders press together per event, whether pressing intensity holds up across the match, and how often pressure actually wins the ball back. These are the numbers most worth handing to an LLM downstream -- volume alone ("pressed for 32% of the time") says little without effectiveness attached.

In [ ]:
# --- co-press count per event: lone chaser vs coordinated press ---
copress_records = []
for team_id, grp in team_pressure_events.groupby("team"):
    team_frames = tpf[tpf["team"] == team_id].sort_values("frame_idx")
    fidx = team_frames["frame_idx"].values
    npress = team_frames["n_pressing"].values
    starts = np.searchsorted(fidx, grp["start_frame"].values, side="left")
    ends = np.searchsorted(fidx, grp["end_frame"].values, side="right")
    for (idx, row), s, e in zip(grp.iterrows(), starts, ends):
        seg = npress[s:e]
        copress_records.append({
            "event_index": idx,
            "max_n_pressing": seg.max() if len(seg) else np.nan,
            "mean_n_pressing_event": seg.mean() if len(seg) else np.nan,
        })

copress_df = pd.DataFrame(copress_records).set_index("event_index")
team_pressure_events = (
    team_pressure_events.drop(columns=["max_n_pressing", "mean_n_pressing_event"], errors="ignore")  # safe to re-run
    .join(copress_df)
)

print("max simultaneous pressers per event, by team:")
print(team_pressure_events.groupby("team")["max_n_pressing"].describe())

print("\nshare of events that were a coordinated press (max_n_pressing >= 2), by team:")
print(team_pressure_events.groupby("team")["max_n_pressing"].apply(lambda s: (s >= 2).mean() * 100).round(1))

In [ ]:
# --- pressing intensity over match time (5-minute bins), chronologically sorted, stoppages excluded ---
BIN_MINUTES = 5
KICKOFF_FRAME = 0   # set to the first-half kickoff frame if the clip starts before kick-off; earlier frames are dropped

tpf_binned = tpf.merge(stoppage_frame_lookup.rename("is_stoppage"), left_on="frame_idx", right_index=True, how="left")
tpf_binned["is_stoppage"] = tpf_binned["is_stoppage"].fillna(False)
tpf_binned = tpf_binned[~tpf_binned["is_stoppage"]].copy()
tpf_binned = tpf_binned[tpf_binned["frame_idx"] >= KICKOFF_FRAME].copy()
tpf_binned["time_bin"] = ((tpf_binned["frame_idx"] - KICKOFF_FRAME) / FPS // (BIN_MINUTES * 60)).astype(int)

pressing_by_bin = (
    tpf_binned.groupby(["team", "time_bin"])["under_team_pressure"]
    .mean()
    .reset_index()
    .rename(columns={"under_team_pressure": "pct_under_pressure"})
)
pressing_by_bin["pct_under_pressure"] = (pressing_by_bin["pct_under_pressure"] * 100).round(1)
pressing_by_bin["bin_label"] = pressing_by_bin["time_bin"].apply(
    lambda b: f"{b * BIN_MINUTES}-{(b + 1) * BIN_MINUTES}min"
)

pressing_intensity_timeline = (
    pressing_by_bin
    .sort_values("time_bin")
    .pivot(index="bin_label", columns="team", values="pct_under_pressure")
)
ordered_labels = pressing_by_bin.sort_values("time_bin")["bin_label"].unique()
pressing_intensity_timeline = pressing_intensity_timeline.loc[ordered_labels]

print("pressing intensity (% of out-of-possession time under pressure) by 5-minute bin, stoppages excluded:")
print(pressing_intensity_timeline)

In [ ]:
# --- mask low-sample bins: a bin with too few frames lets one short sequence swing the average ---
MIN_BIN_FRAMES = 1000   # 40 s at 25 fps; a full 5-min bin has up to 7,500 frames, ~2-3k for the defending team

# rebuild fresh from pressing_by_bin every time -- avoids mutating a possibly-stale table
# if this cell is re-run more than once or out of order
pressing_intensity_timeline = (
    pressing_by_bin
    .sort_values("time_bin")
    .pivot(index="bin_label", columns="team", values="pct_under_pressure")
)
ordered_labels = pressing_by_bin.sort_values("time_bin")["bin_label"].unique()
pressing_intensity_timeline = pressing_intensity_timeline.loc[ordered_labels]

bin_frame_counts = tpf_binned.groupby(["team", "time_bin"]).size().reset_index(name="n_frames")
bin_frame_counts["bin_label"] = bin_frame_counts["time_bin"].apply(
    lambda b: f"{b * BIN_MINUTES}-{(b + 1) * BIN_MINUTES}min"
)

low_sample = bin_frame_counts[bin_frame_counts["n_frames"] < MIN_BIN_FRAMES]

for _, row in low_sample.iterrows():
    pressing_intensity_timeline.loc[row["bin_label"], row["team"]] = np.nan

print(f"masked {len(low_sample)} team/bin cell(s) below MIN_BIN_FRAMES={MIN_BIN_FRAMES} (unreliable sample size):")
print(low_sample[["bin_label", "team", "n_frames"]])

print("\npressing intensity timeline, low-sample bins masked as NaN:")
print(pressing_intensity_timeline)

In [ ]:
# --- pressure -> regain conversion rate ---
MAX_REGAIN_SEC = 5.0
MAX_REGAIN_FRAMES = int(MAX_REGAIN_SEC * FPS)

def regained_within_window(team_id, end_frame):
    # stable hold (>= MIN_HOLD_SEC) starting at or after the event's last frame, within MAX_REGAIN_SEC
    return stable_regain(team_id, end_frame - 1, end_frame + MAX_REGAIN_FRAMES)

team_pressure_events["regained_ball"] = team_pressure_events.apply(
    lambda r: regained_within_window(r["team"], r["end_frame"]), axis=1
)

regain_summary = (
    team_pressure_events.groupby("team")["regained_ball"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "regain_rate"})
)
regain_summary["regain_rate"] = (regain_summary["regain_rate"] * 100).round(1)
print(f"pressure -> regain conversion rate (ball won back by the pressing team within {MAX_REGAIN_SEC}s of the event ending):")
print(regain_summary)

## Hand-off
Saves what the later notebooks need.

In [ ]:
save_state(globals(), [
    'COUNTER_PRESS_WINDOW_FRAMES',
    'MAX_REGAIN_SEC',
    'ZONE_WEIGHTS',
    'final_pressure_by_team',
    'possession_events',
    'press_lookup',
    'pressing_intensity_timeline',
    'regain_summary',
    'counter_press_summary',
    'counter_press_landmark',
    'pressing_pct_reconciliation',
    'team_pressure_events',
    'turnovers_annotated',
    'weighted_pressure_seconds',
], notebook='03_timing_and_possession.ipynb')